# Exceptions API Reference

Developer-facing statements defined in `libs/core/langchain_core/exceptions.py`.

# `LangChainException: Exception`

General LangChain exception.

---

# `TracerException: LangChainException`

Base class for exceptions raised by tracer functionality.

---

# `OutputParserException: ValueError, LangChainException`

Signals an output-parsing failure so it can be handled separately from other execution errors.

## Fields

```python
observation: str | None # Explanation that can be passed back to a model for remediation
llm_output: str | None # Model output that failed to parse
send_to_llm: bool # Whether the observation and failed output should be returned to an agent
```

## Constructor

```python
OutputParserException(
    error: Any, # Exception or error message being re-raised
    observation: str | None = None, # Explanation that can help a model correct the output
    llm_output: str | None = None, # Model output that caused the parsing failure
    send_to_llm: bool = False, # Whether to return the remediation context to an agent
)
```

When `error` is a string, it is converted with `create_message` using `ErrorCode.OUTPUT_PARSING_FAILURE`.

Raises `ValueError` when `send_to_llm` is `True` and either `observation` or `llm_output` is not provided.

---

# `ContextOverflowError: LangChainException`

Raised by chat models when input tokens exceed the model's supported context window.

---

# `ErrorCode: Enum`

Error codes used to construct troubleshooting links.

```python
INVALID_PROMPT_INPUT = "INVALID_PROMPT_INPUT"
INVALID_TOOL_RESULTS = "INVALID_TOOL_RESULTS"
MESSAGE_COERCION_FAILURE = "MESSAGE_COERCION_FAILURE"
MODEL_AUTHENTICATION = "MODEL_AUTHENTICATION"
MODEL_NOT_FOUND = "MODEL_NOT_FOUND"
MODEL_RATE_LIMIT = "MODEL_RATE_LIMIT"
OUTPUT_PARSING_FAILURE = "OUTPUT_PARSING_FAILURE"
```

---

# `create_message`

Creates an error message containing a link to the LangChain troubleshooting guide for the supplied error code.

```python
create_message(
    *,
    message: str, # Error message to display
    error_code: ErrorCode, # Error code appended to the troubleshooting URL
) -> str # Message followed by the troubleshooting link
```

In [ ]:
# Use case: Handle invalid LLM output during parsing
from langchain_core.exceptions import OutputParserException # Import the parsing exception
from langchain_core.output_parsers import JsonOutputParser # Import LangChain's JSON output parser


parser = JsonOutputParser() # Create a parser that expects valid JSON output
llm_output = "The answer is Python." # Simulate invalid non-JSON model output

try: # Start exception handling
    parsed_result = parser.parse(llm_output) # Try to parse the model output as JSON
    print(parsed_result) # Display the parsed result when parsing succeeds

except Exception as error: # Catch the original parsing error
    parsing_error = OutputParserException( # Create a LangChain-specific parsing exception
        error=error, # Store the original parsing error
        observation="Return only valid JSON.", # Tell the agent how the response should be corrected
        llm_output=llm_output, # Store the model output that failed
        send_to_llm=True, # Allow the correction details to be sent back to the model
    ) # Finish creating the parsing exception

    print("Error:", parsing_error) # Display the original parsing error
    print("Observation:", parsing_error.observation) # Display the correction instruction
    print("LLM output:", parsing_error.llm_output) # Display the failed model output
    print("Send to LLM:", parsing_error.send_to_llm) # Display whether retry information should be sent

In [ ]:
# Use case: Generate a troubleshooting message
from langchain_core.exceptions import ErrorCode, create_message # Import the error-code enum and message helper


error_message = create_message( # Create an error message with a troubleshooting link
    message="The model output could not be parsed.", # Provide the main error description
    error_code=ErrorCode.OUTPUT_PARSING_FAILURE, # Select the relevant LangChain error code
) # Finish creating the message

print(error_message) # Display the error message and troubleshooting URL

In [ ]:
# Use case: Catch a context-window overflow
from langchain_core.exceptions import ContextOverflowError # Import the context-overflow exception


def send_large_prompt(token_count: int, context_limit: int) -> None: # Simulate sending a prompt to a model
    if token_count > context_limit: # Check whether the prompt exceeds the model limit
        raise ContextOverflowError("The input exceeds the model context window.") # Raise the LangChain exception

    print("Prompt accepted.") # Continue when the prompt fits within the limit


try: # Start exception handling
    send_large_prompt(token_count=15000, context_limit=8000) # Simulate an oversized model input

except ContextOverflowError as error: # Catch only context-window overflow errors
    print(error) # Display the context-overflow message